In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import pandas as pd
from tqdm import tqdm
import os
from transformers import pipeline
import torch

model_name = "Qwen/Qwen3-8B"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype="auto", device_map="auto"
)


def generate_answer(problem):
    messages = [{"role": "user", "content": problem}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,  # Switches between thinking and non-thinking modes. Default is True.
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # conduct text completion
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=32768,
        temperature=0.6,
        top_p=0.95,
        top_k=20,
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]) :].tolist()

    # parsing thinking content
    try:
        # rindex finding 151668 (</think>)
        index = len(output_ids) - output_ids[::-1].index(151668)
    except ValueError:
        index = 0

    content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")
    return content

In [ ]:
dataset = load_dataset("large-traversaal/mgsm_urdu_final", split="test").select(
    range(3)
)

results = []

for row in tqdm(dataset):
    response = generate_answer(row["urdu_question"])  # Your model response

    results.append(
        {
            "question": row["urdu_question"],
            "answer": row["answer_number"],
            "response": response,
        }
    )
    print(row["question"])

df = pd.DataFrame(results)
df.to_csv("qwen3-8b-response.csv", index=False)

In [ ]:
df = pd.read_csv("qwen3-8b-response.csv")

validation_results = []

model_id = "openai/gpt-oss-20b"

pipe = pipeline(
    "text-generation",
    model=model_id,
    torch_dtype="auto",
    device_map="auto",
)

for _, row in df.iterrows():

    prompt = f"""You are an answer validator for Urdu math reasoning problems.
    
    Input:
    
    * Question: {row['question']}
    * Solution: {row['response']}
    * Expected Answer: {row['answer']}
    
    Task:
    
    1. Read the Urdu math problem and the provided solution.
    2. Determine the final answer produced by the solution.
    3. Compare it with the Expected Answer.
    4. Treat numerically equivalent values as equal:
    
       * 2 = 2.0 = 2.00
       * 5 = 5.000
       * Ignore insignificant trailing zeros in decimals.
    5. If both answers represent the same numeric value, return:
       True
    6. Otherwise return:
       False
    
    Rules:
    
    * Return ONLY one word: True or False.
    * Do not provide explanations, reasoning, calculations, or extra text.
    * Comparison should be based on the final answer only.
    * For numeric answers, compare by value, not string format.
    * If the solution's final answer cannot be determined with confidence, return False.
     """

    messages = [{"role": "user", "content": prompt}]
    outputs = pipe(messages, max_new_tokens=1000)
    result = (
        True
        if "True.assistantfinalTrue" in outputs[0]["generated_text"][-1]["content"]
        else False
    )
    validation_results.append(result)

df["is_correct"] = validation_results

df.to_csv("qwen3-8b_correct.csv", index=False)

print(df.head())